# Budget Forcing

**Paper**: [s1: Simple test-time scaling](https://arxiv.org/abs/2501.19393)

**Authors**: Niklas Muennighoff, Zitong Yang, Weijia Shi, Xiang Lisa Li, Li Fei-Fei, Hannaneh Hajishirzi, Luke Zettlemoyer, Percy Liang, Emmanuel Candes, Tatsunori Hashimoto

Budget forcing controls the length of a reasoning model's thinking at test time. It caps the thinking phase at a token budget and can either shorten reasoning (force the closing think tag once the budget is hit) or lengthen it (append an extension such as "Wait" to prompt continued reasoning) before generating the final answer.

Budget forcing is a decoding driver built on the generic phased driver: a bounded thinking phase, optional extension rounds, a forced closing tag, and an unbounded answer phase.

The method assumes a reasoning model. The thinking-phase boundary is the model's own closing think tag, so the driver can only find that boundary if the model actually emits one; on a non-reasoning model the tag never appears and the method degenerates to blind truncation plus a pasted-in tag.

## Method parameters

| parameter | type | description |
| --------- | ---- | ----------- |
| `max_thinking_tokens` | `int` | Token budget for each thinking segment |
| `extension_text` | `str` | Text appended to prolong reasoning |
| `num_extensions` | `int` | Number of extension rounds (0 disables) |
| `end_think` | `str` | The closing-think marker |

## Setup

If running this from a Google Colab notebook, uncomment the clone cell below. It is not necessary when running from a virtual environment where the package is already installed.

In [1]:
# !git clone https://github.com/IBM/AISteer360.git
# %cd AISteer360

The following authentication steps may be necessary to access any gated models (after being granted access by Hugging Face). Uncomment the following if you need to log in to the Hugging Face Hub:

In [ ]:
# !pip install -q python-dotenv
# from dotenv import load_dotenv
# import os

# load_dotenv()
# token = os.getenv("HUGGINGFACE_TOKEN")
# from huggingface_hub import login
# login(token=token)

## Example: dialing a reasoning model's thinking budget

We use `deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B`, a small open reasoning model. Its chat template opens the thinking block (the prompt ends with `<think>`), and the model closes it by emitting `</think>` before writing its final answer, so the driver's boundary marker occurs naturally in every generation. Following the model card we sample with temperature 0.6 and top-p 0.95 rather than decoding greedily, with a fixed seed so runs are comparable.

In [ ]:
import re

from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

from aisteer360.algorithms.core.steering_pipeline import SteeringPipeline
from aisteer360.algorithms.output_control.budget_forcing.control import BudgetForcing

MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
END_THINK = "</think>"
SAMPLING = {"do_sample": True, "temperature": 0.6, "top_p": 0.95}

Full reasoning streams are long, so two small helpers keep the outputs readable: one splits a generation into its thinking span and final answer, the other counts thinking tokens. The split is on the first closing tag, so whatever the model generates after the (possibly forced) tag counts as answer, and when a generation runs out of tokens before any tag appears, the whole stream counts as thinking.

In [4]:
def split_thinking(text: str, end_think: str = END_THINK) -> tuple[str, str]:
    if end_think in text:
        thinking, answer = text.split(end_think, 1)
        return thinking, answer.strip()
    return text, ""


def num_tokens(tokenizer, text: str) -> int:
    return len(tokenizer(text, add_special_tokens=False)["input_ids"])

### Baseline: the model's natural thinking length

First, how the model behaves unforced. We generate with a plain `model.generate` call and a generous token limit, then measure how long the model chooses to think on a short multi-step word problem (the correct answer is 5).

In [5]:
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map="auto", dtype="auto")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

prompt = (
    "Betty is saving money for a new wallet which costs $100. Betty has only half of the money she needs. "
    "Her parents give her $15 for that purpose, and her grandparents give twice as much as her parents. "
    "How much more money does Betty need, in dollars?"
)
chat = tokenizer.apply_chat_template(
    [{"role": "user", "content": prompt}],
    tokenize=False,
    add_generation_prompt=True,
)
inputs = tokenizer(chat, return_tensors="pt", add_special_tokens=False).to(model.device)

set_seed(42)
baseline_ids = model.generate(**inputs, max_new_tokens=2048, pad_token_id=tokenizer.eos_token_id, **SAMPLING)
baseline_text = tokenizer.decode(baseline_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

thinking, answer = split_thinking(baseline_text)
print(f"thinking tokens: {num_tokens(tokenizer, thinking)}")
print(f"\nanswer: {answer}")

thinking tokens: 784

answer: Betty needs $100 for a new wallet. She currently has half of that amount, which is $50. Her parents give her $15, and her grandparents give her twice as much as her parents, which is $30.

1. Betty's current savings: $50
2. After her parents give her $15: $50 + $15 = $65
3. After her grandparents give her $30: $65 + $30 = $95

The amount Betty still needs is $100 - $95 = $5.

\[
\boxed{5}
\]


Left alone, the model spends a substantial thinking budget on this problem before committing to an answer. That natural length is the reference point for everything below: shortening means cutting below it, extending means pushing past where the model would have stopped.

### Shortening: cap the budget and force the tag

We cap thinking at `max_thinking_tokens=64` with no extensions. The plan is a thinking phase that stops at the closing tag or at 64 tokens (whichever comes first), the forced `</think>`, then the answer phase. At 64 tokens the model is still mid-thought, so the thinking span below ends abruptly where the tag was pasted in. This is the "shorten" half of s1.

In [6]:
budget_forcing = BudgetForcing(max_thinking_tokens=64, num_extensions=0, end_think=END_THINK)

pipeline = SteeringPipeline(
    model_name_or_path=MODEL_NAME,
    controls=[budget_forcing],
    device_map="auto",
    hf_model_kwargs={"dtype": "auto"},
)
pipeline.steer()

set_seed(42)
output = pipeline.generate(
    input_ids=inputs["input_ids"].to(pipeline.model.device),
    max_new_tokens=512,
    pad_token_id=tokenizer.eos_token_id,
    **SAMPLING,
)
forced_text = tokenizer.decode(output[0], skip_special_tokens=True)
thinking, answer = split_thinking(forced_text)

print(f"thinking tokens: {num_tokens(tokenizer, thinking)}")
print(f"\nend of thinking span: ...{thinking[-160:]}")
print(f"\nanswer: {answer}")

You're using a LlamaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


thinking tokens: 64

end of thinking span: ...et me figure out how much she currently has and how much more she needs. 

First, the problem says Betty has only half of the money she needs. So, if the wallet

answer: To determine how much more money Betty needs, let's break down her current savings.

1. The total cost of the wallet is $100.
2. Betty has half of what she needs, which is half of $100, so that's $50.
3. Her parents give her $15, and her grandparents give twice as much as her parents. Since her parents give $15, her grandparents give 2 × $15 = $30.
4. Adding her parents' and grandparents' contributions: $15 + $30 = $45.
5. Now, Betty has her own $50 plus her parents' and grandparents' $45, totaling $50 + $45 = $95.
6. Finally, subtracting the total she has ($95) from the cost of the wallet ($100) gives her the amount she still needs: $100 - $95 = $5.

So, Betty needs an additional $5 to buy the wallet.
</think>

To determine how much more money Betty needs, let's break down

The thinking span stops mid-sentence at exactly the budget, and the model is forced to answer from whatever partial reasoning it has. Notice how the model compensates: the "answer" it writes after the forced tag quietly re-derives the whole solution instead of trusting the truncated thought. Cutting the thinking budget moved the reasoning; it did not remove it.

### Extending: append "Wait" and keep thinking

Extensions are the "lengthen" half of s1. Each extension round appends `Wait` to the stream and opens another bounded thinking segment, so a thought the budget would have cut short gets prolonged instead. We keep the per-segment budget at 128 tokens so the splice points are easy to locate: the driver appends `Wait` right after tokens 128 and 257 of the continuation.

In [7]:
budget_forcing = BudgetForcing(
    max_thinking_tokens=128,
    extension_text="Wait",
    num_extensions=2,
    end_think=END_THINK,
)

pipeline = SteeringPipeline(
    model_name_or_path=MODEL_NAME,
    controls=[budget_forcing],
    device_map="auto",
    hf_model_kwargs={"dtype": "auto"},
)
pipeline.steer()

set_seed(42)
output = pipeline.generate(
    input_ids=inputs["input_ids"].to(pipeline.model.device),
    max_new_tokens=512,
    pad_token_id=tokenizer.eos_token_id,
    **SAMPLING,
)
extended_text = tokenizer.decode(output[0], skip_special_tokens=True)
thinking, answer = split_thinking(extended_text)
print(f"thinking tokens: {num_tokens(tokenizer, thinking)}")

wait_len = num_tokens(tokenizer, "Wait")
out_ids = output[0]
for i, splice_at in enumerate([128, 128 + wait_len + 128], start=1):
    window = tokenizer.decode(out_ids[splice_at - 20:splice_at + wait_len + 20], skip_special_tokens=True)
    print(f"\nsplice {i}: ...{window}...")

print(f"\nanswer: {answer}")

You're using a LlamaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


thinking tokens: 386

splice 1: ...0 is 50. So, Betty currently has $50. That makes sense because ifWait, no, wait, she has half of the money she needs. So, maybe I should think...

splice 2: ...15 is $65. Got that. So, after her parents' contribution, she hasWait, no, wait, she already had $50 and she gets $15 more, so...

answer: Betty needs a total of $100 for the wallet. She currently has half of this amount, which is $50. Her parents contribute $15, bringing her total to $65. Her grandparents then give her twice the amount her parents contributed, which is $30. Adding this to her current total, Betty now has $65 + $30 = $95. 

To find out how much more money Betty needs, subtract the amount she currently has ($95) from the total cost ($100). So, she needs $5 more.

$\boxed{5}$


Each splice shows the same pattern: the segment is cut mid-thought at its budget, the appended `Wait` lands, and the model picks the reasoning back up, often by re-examining what it had just concluded. The total thinking length is now set by the driver, not by when the model felt done.

### The s1 story: answer quality vs. thinking budget

Budget forcing is the mechanism behind s1's test-time scaling curves, where answer quality is a function of allotted thinking compute. The sweep below runs the same problem at three budgets and tabulates the thinking tokens actually used and the final answer.

In [8]:
results = []
for budget in [64, 256, 1024]:
    sweep_pipeline = SteeringPipeline(
        model_name_or_path=MODEL_NAME,
        controls=[BudgetForcing(max_thinking_tokens=budget, num_extensions=0, end_think=END_THINK)],
        device_map="auto",
        hf_model_kwargs={"dtype": "auto"},
    )
    sweep_pipeline.steer()
    set_seed(42)
    output = sweep_pipeline.generate(
        input_ids=inputs["input_ids"].to(sweep_pipeline.model.device),
        max_new_tokens=max(512, budget),
        pad_token_id=tokenizer.eos_token_id,
        **SAMPLING,
    )
    thinking, answer = split_thinking(tokenizer.decode(output[0], skip_special_tokens=True))
    numbers = re.findall(r"-?\d+", answer)
    results.append((budget, num_tokens(tokenizer, thinking), numbers[-1] if numbers else answer[:40]))

print(f"{'budget':>7}  {'thinking tokens':>16}  {'final answer':>13}")
for budget, used, final_answer in results:
    print(f"{budget:>7}  {used:>16}  {final_answer:>13}")

You're using a LlamaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


You're using a LlamaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


You're using a LlamaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


 budget   thinking tokens   final answer
     64                64              5
    256               256             35
   1024               784              5


The thinking-token column tracks the budget exactly, which is the compute half of the s1 curve. On this problem the answer half is flat: every budget lands on the correct value, because when thinking is cut hard the model finishes the derivation inside its answer phase instead (visible in the shortened run above). Extra budget here buys directness rather than correctness. On problems at the edge of the model's ability, the same dial moves accuracy, which is the s1 result.

### Mechanics

`BudgetForcing` is a preset of the generic phased driver. Its per-example plan is:

- `Generated(until="</think>", budget=max_thinking_tokens)`, the bounded thinking phase
- `num_extensions` repetitions of `Fixed(extension_text)` followed by another bounded `Generated`
- `Fixed("</think>")`, the forced closing tag
- `Generated()`, the unbounded answer phase

Phase boundaries are substring stops on the closing marker only; the opening `<think>` plays no role in the mechanics (here it lives in the prompt, courtesy of the chat template). `Fixed` phases are plain token appends, so when the model closes its thinking naturally within budget, the forced tag still lands and the stream carries the tag twice. Plans are built per example, and batched inputs are handled by looping over rows. Every `Generated` phase delegates to `model.generate` with the pipeline's composed stacks, so a step-level control (for example RAD) steers each phase, including the extensions.

### Takeaway

Budget forcing turns thinking length into an inference-time dial: one integer trades answer quality against decode compute, and the "Wait" trick buys extra reasoning on demand without touching weights or prompts. It only makes sense on models that already externalize their reasoning between think tags.

[phased_decoding.ipynb](../generics/phased_decoding.ipynb) demonstrates the generic this preset is built on, including a thinking-intervention plan that splices steering text into the reasoning stream rather than bounding its length. See the [output control](https://ibm.github.io/AISteer360/concepts/controls/#output-control) section of the docs for the full family.